# 🟡 Day 6 — Tools & Your First Agent

**LangGraph from Zero to Advanced · Day 6 of 16**

**Today's goal:** give the LLM **tools** (Python functions it can ask to run) and build a real **agent** — a loop where the LLM thinks, uses tools, and answers.

| Lesson | Topic |
|---|---|
| 1 | What is a tool? (`@tool`) |
| 2 | `bind_tools`: the LLM *asks* for a tool |
| 3 | Run the tool by hand (`ToolMessage`) |
| 4 | Build the agent loop yourself |
| 5 | Same agent with `ToolNode` + `tools_condition` |
| 6 | Tool errors don't crash the agent |
| 7 | The one-line agent: `create_agent` |
| 8 | Mini project: Travel helper agent |
| 9 | Practice |

In [ ]:
%pip install -q -U langgraph langchain langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)

def show_graph(app):
    """Draw the graph as a picture, or print it as text when offline."""
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

print("✅ Ready")

## Lesson 1 — What is a tool?

LLMs are bad at exact maths and don't know live data (weather, your database…). A **tool** is a normal Python function the LLM can ask you to run.

Add the `@tool` decorator. LangChain reads the **function name**, **type hints** and **docstring** and shows them to the LLM — so write a clear docstring!

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers. Use this for any multiplication."""
    return a * b

@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

print("name:", multiply.name)
print("description:", multiply.description)
print("args:", multiply.args)

# A tool can still be run directly:
print("result:", multiply.invoke({"a": 12, "b": 7}))

## Lesson 2 — `bind_tools`: the LLM *asks* for a tool

`llm.bind_tools([...])` tells the model which tools exist. The model **does not run** the tool — it returns an `AIMessage` with **`tool_calls`**: "please run `multiply` with a=123, b=45".

In [ ]:
tools = [multiply, add]
llm_with_tools = llm.bind_tools(tools)

ai_msg = llm_with_tools.invoke("What is 123 multiplied by 45?")
print("content   :", repr(ai_msg.content))
print("tool_calls:", ai_msg.tool_calls)

In [ ]:
# A normal question → no tool needed → normal answer
plain = llm_with_tools.invoke("Say hi in Hindi.")
print("tool_calls:", plain.tool_calls, "| content:", plain.content)

## Lesson 3 — Run the tool by hand

The full round-trip:

1. 👤 user asks → 2. 🤖 AI returns `tool_calls` → 3. 🔧 **we** run the tool and send back a **`ToolMessage`** (with the same `tool_call_id`) → 4. 🤖 AI writes the final answer.

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

tools_by_name = {t.name: t for t in tools}

messages = [HumanMessage(content="What is 123 multiplied by 45?")]
ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)

for call in ai_msg.tool_calls:
    output = tools_by_name[call["name"]].invoke(call["args"])
    print(f"🔧 ran {call['name']}({call['args']}) = {output}")
    messages.append(ToolMessage(content=str(output), tool_call_id=call["id"]))

final = llm_with_tools.invoke(messages)
print("🤖", final.content)

## Lesson 4 — Build the agent loop yourself

Some questions need **several** tool calls (e.g. *"(3 + 5) × 12"*). So we loop until the AI stops asking for tools. That loop **is** an agent (called *ReAct*: Reason + Act).

```
START ──► agent ──(tool_calls?)──yes──► tools ──┐
            ▲                                   │
            └───────────────────────────────────┘
               │ no
               ▼
              END
```

In [ ]:
from typing import Literal
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_core.messages import SystemMessage

SYSTEM = SystemMessage(content="You are a maths helper. Always use tools for arithmetic.")

def agent(state: MessagesState) -> dict:
    return {"messages": [llm_with_tools.invoke([SYSTEM] + state["messages"])]}

def run_tools(state: MessagesState) -> dict:
    last = state["messages"][-1]
    results = []
    for call in last.tool_calls:
        output = tools_by_name[call["name"]].invoke(call["args"])
        results.append(ToolMessage(content=str(output), tool_call_id=call["id"]))
    return {"messages": results}

def should_continue(state: MessagesState) -> Literal["run_tools", "__end__"]:
    return "run_tools" if state["messages"][-1].tool_calls else END

g = StateGraph(MessagesState)
g.add_node("agent", agent)
g.add_node("run_tools", run_tools)
g.add_edge(START, "agent")
g.add_conditional_edges("agent", should_continue)
g.add_edge("run_tools", "agent")          # 👈 the loop back
my_agent = g.compile()
show_graph(my_agent)

In [ ]:
result = my_agent.invoke({"messages": [("user", "What is (3 + 5) multiplied by 12?")]})
for m in result["messages"]:
    m.pretty_print()

## Lesson 5 — Same agent with `ToolNode` + `tools_condition`

This pattern is so common that LangGraph ships both pieces:

* **`ToolNode(tools)`** — runs every tool call in the last message (in parallel) and returns `ToolMessage`s.
* **`tools_condition`** — router: goes to `"tools"` if there are tool calls, else `END`.

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition

g = StateGraph(MessagesState)
g.add_node("agent", agent)
g.add_node("tools", ToolNode(tools))      # node MUST be named "tools" for tools_condition
g.add_edge(START, "agent")
g.add_conditional_edges("agent", tools_condition)
g.add_edge("tools", "agent")
agent_app = g.compile()
show_graph(agent_app)

In [ ]:
for step in agent_app.stream({"messages": [("user", "Add 2.5 and 7.5, then multiply the result by 4")]},
                             stream_mode="updates"):
    for node, update in step.items():
        for m in update["messages"]:
            print(f"[{node}]", m.type, "|", m.tool_calls if getattr(m, "tool_calls", None) else m.content)

## Lesson 6 — Tool errors don't crash the agent

What if a tool raises an exception (e.g. divide by zero)? By default the error **stops the graph**. With `ToolNode(tools, handle_tool_errors=True)` the error is sent back to the LLM as a `ToolMessage` instead — the LLM can then fix its mistake or explain it to the user.

In [ ]:
from langchain_core.messages import AIMessage

@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

# A tiny graph with just the tool node
g = StateGraph(MessagesState)
g.add_node("tools", ToolNode([divide], handle_tool_errors=True))
g.add_edge(START, "tools")
g.add_edge("tools", END)
safe_tools = g.compile()

# Pretend the AI asked to divide by zero:
bad_request = AIMessage(content="", tool_calls=[{"name": "divide", "args": {"a": 1, "b": 0}, "id": "call_1"}])
result = safe_tools.invoke({"messages": [bad_request]})
print(result["messages"][-1].content)     # the error comes back as text — no crash

## Lesson 7 — The one-line agent: `create_agent`

Once you understand the loop, you can use LangChain's prebuilt agent. It builds the **same graph** (agent ⇄ tools) for you — and it *is* a LangGraph graph, so `invoke`, `stream`, memory, etc. all work.

In [ ]:
from langchain.agents import create_agent

quick_agent = create_agent(
    model=llm,
    tools=[multiply, add, divide],
    system_prompt="You are a helpful maths assistant. Use tools for every calculation.",
)
show_graph(quick_agent)

res = quick_agent.invoke({"messages": [("user", "What is 144 divided by 12, plus 10?")]})
print(res["messages"][-1].content)

💡 **When to use what?** Learn with the hand-made graph (Lessons 4–5). Use `create_agent` for quick standard agents. Build your own graph when you need custom steps, routing, or several agents (Days 10–16).

## Lesson 8 — Mini project: Travel helper agent

Tools return **fake data** so it's free and predictable — in real life they would call weather/currency APIs.

In [ ]:
@tool
def get_weather(city: str) -> str:
    """Get today's weather for a city."""
    data = {"mumbai": "31°C, humid, light rain", "delhi": "36°C, sunny",
            "bengaluru": "24°C, pleasant", "london": "14°C, cloudy"}
    return data.get(city.lower(), f"No weather data for {city}")

@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert money between INR, USD, EUR and GBP."""
    to_inr = {"INR": 1, "USD": 83, "EUR": 90, "GBP": 105}
    f, t = from_currency.upper(), to_currency.upper()
    if f not in to_inr or t not in to_inr:
        return "Unsupported currency"
    return f"{amount} {f} = {round(amount * to_inr[f] / to_inr[t], 2)} {t}"

@tool
def packing_tip(weather: str) -> str:
    """Suggest what to pack based on a weather description."""
    w = weather.lower()
    if "rain" in w:  return "Pack an umbrella and quick-dry clothes."
    if "sunny" in w: return "Pack sunscreen, sunglasses and a water bottle."
    if "cloud" in w: return "Pack a light jacket."
    return "Pack comfortable clothes."

travel_tools = [get_weather, convert_currency, packing_tip]
travel_llm = llm.bind_tools(travel_tools)
TRAVEL_SYSTEM = SystemMessage(content="You are a friendly travel assistant. Use tools; never guess weather or rates.")

def travel_agent(state: MessagesState) -> dict:
    return {"messages": [travel_llm.invoke([TRAVEL_SYSTEM] + state["messages"])]}

g = StateGraph(MessagesState)
g.add_node("agent", travel_agent)
g.add_node("tools", ToolNode(travel_tools))
g.add_edge(START, "agent")
g.add_conditional_edges("agent", tools_condition)
g.add_edge("tools", "agent")
travel = g.compile()

In [ ]:
q = "I'm going to London with 50000 INR. What's the weather, what should I pack, and how many GBP is my money?"
res = travel.invoke({"messages": [("user", q)]})
for m in res["messages"]:
    if m.type == "ai" and m.tool_calls:
        print("🔧 asked for:", [c["name"] for c in m.tool_calls])
print("\n🤖", res["messages"][-1].content)

## Lesson 9 — Practice

**Exercise 1.** Add a tool `word_count(text: str) -> int` to `quick_agent`'s tools and ask: *"How many words are in 'LangGraph makes agents easy to build'?"*

**Exercise 2.** Add a `max_tool_rounds` safety: extend the state with `rounds: int`, increment it in the agent node, and stop (go to `END`) after 3 rounds even if the LLM still wants tools.

In [ ]:
# ✅ Solution 1
@tool
def word_count(text: str) -> int:
    """Count the words in a text."""
    return len(text.split())

wc_agent = create_agent(model=llm, tools=[word_count])
print(wc_agent.invoke({"messages": [("user", "How many words are in 'LangGraph makes agents easy to build'?")]})["messages"][-1].content)

In [ ]:
# ✅ Solution 2
class SafeState(MessagesState):
    rounds: int

def safe_agent(state: SafeState) -> dict:
    return {"messages": [llm_with_tools.invoke([SYSTEM] + state["messages"])],
            "rounds": state.get("rounds", 0) + 1}

def safe_route(state: SafeState) -> Literal["tools", "__end__"]:
    if state["rounds"] >= 3:
        return END
    return "tools" if state["messages"][-1].tool_calls else END

g = StateGraph(SafeState)
g.add_node("agent", safe_agent)
g.add_node("tools", ToolNode(tools))
g.add_edge(START, "agent")
g.add_conditional_edges("agent", safe_route)
g.add_edge("tools", "agent")
res = g.compile().invoke({"messages": [("user", "Multiply 6 by 7")], "rounds": 0})
print("rounds used:", res["rounds"], "|", res["messages"][-1].content)
assert res["rounds"] <= 3
print("✅ All exercises solved!")

## 🎯 Day 6 Recap

* `@tool` turns a function into a tool — name, type hints and docstring matter.
* `llm.bind_tools(tools)` → the LLM returns `tool_calls`; **your code** runs them and replies with `ToolMessage`s.
* Agent = loop: `agent → tools → agent …` until no more tool calls.
* `ToolNode` + `tools_condition` = the prebuilt pieces; `create_agent` = the whole agent in one line.

**Tomorrow (Day 7):** give your agents **memory** with checkpointers and `thread_id`.